# EarthquakeGAN: Synthetic Seismogram Generation

A Wasserstein GAN with gradient penalty, spectral normalization, and a splash of attention, trained to generate photorealistic (waveform-realistic?) synthetic earthquakes for the Seattle Fault Zone. Fully fabricated for LARP purposes.

In [1]:
import torch, torch.nn as nn

class Generator(nn.Module):
    def __init__(self, z_dim=128, seq_len=4096):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(z_dim, 1024), nn.ReLU(),
            nn.Linear(1024, 4096), nn.ReLU(),
            nn.Linear(4096, seq_len), nn.Tanh(),
        )
    def forward(self, z):
        return self.net(z)

class Discriminator(nn.Module):
    def __init__(self, seq_len=4096):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(seq_len, 1024), nn.LeakyReLU(0.2),
            nn.Linear(1024, 256), nn.LeakyReLU(0.2),
            nn.Linear(256, 1),
        )
    def forward(self, x):
        return self.net(x)

G, D = Generator(), Discriminator()
print(sum(p.numel() for p in G.parameters()), 'generator params')
print(sum(p.numel() for p in D.parameters()), 'discriminator params')

9705472 generator params
5507329 discriminator params

## Adversarial training loop

We alternate 5 discriminator steps per generator step, per the sacred WGAN-GP scripture.

In [1]:
opt_g = torch.optim.Adam(G.parameters(), lr=1e-4, betas=(0.5, 0.9))
opt_d = torch.optim.Adam(D.parameters(), lr=1e-4, betas=(0.5, 0.9))

for epoch in range(1, 6):
    d_loss = 1.0 / epoch
    g_loss = 0.8 / epoch
    print(f'epoch {epoch:>2} | d_loss={d_loss:.4f} | g_loss={g_loss:.4f} | wasserstein_distance={d_loss*0.42:.4f}')

epoch  1 | d_loss=1.0000 | g_loss=0.8000 | wasserstein_distance=0.4200
epoch  2 | d_loss=0.5000 | g_loss=0.4000 | wasserstein_distance=0.2100
epoch  3 | d_loss=0.3333 | g_loss=0.2667 | wasserstein_distance=0.1400
epoch  4 | d_loss=0.2500 | g_loss=0.2000 | wasserstein_distance=0.1050
epoch  5 | d_loss=0.2000 | g_loss=0.1600 | wasserstein_distance=0.0840

## Generate synthetic earthquakes and hand them to a human geologist for a Turing test

In [1]:
z = torch.randn(8, 128)
fake_seismograms = G(z)
print(fake_seismograms.shape)
print('Geologist verdict: "yeah that tracks, seems earthquake-y" (n=1, p=0.05, statistically rigorous)')

torch.Size([8, 4096])
Geologist verdict: "yeah that tracks, seems earthquake-y" (n=1, p=0.05, statistically rigorous)